# 06 — Flickr30k Pointing Game: Frozen SigLIP-B/16-384 Baseline

Evaluates the frozen SigLIP-B/16-384 model on the **Pointing Game** benchmark
using Flickr30k Entities val split (200 images).

**Protocol:** For each (phrase, bbox) pair in the val set, encode the phrase,
compute per-patch cosine similarity via MaskCLIP-style last-attention bypass,
find the argmax patch, and check whether its centre falls inside the GT bounding box.

**Metric:** Pointing Game Accuracy (%) = correct / total phrase-bbox pairs.

| Cell | Purpose |
|------|---------|
| 1 | Imports and config |
| 2 | Load Flickr30k parquet + Entities annotations |
| 3 | Load frozen SigLIP-B/16-384 |
| 4 | `pointing_game_eval` function |
| 5 | Run on 200 val images, print results |

In [1]:
# ── 1. Imports and config ──────────────────────────────────────────────────────
import io, os, random, sys, types
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image as PILImage
from tqdm.auto import tqdm
from datasets import load_dataset
from transformers import SiglipModel, AutoProcessor

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

CFG = dict(
    model_id   = 'google/siglip-base-patch16-384',
    eval_size  = 384,
    patch_size = 16,
    n_images   = 200,   # val images to evaluate
    seed       = 42,
)
CFG['n_side'] = CFG['eval_size'] // CFG['patch_size']  # 24

# Paths — annotations are already downloaded locally
REPO_ROOT    = Path('..') if Path('../notebooks').exists() else Path('.')
ENTITIES_DIR = Path('data/flickr30k_entities')
ANN_DIR      = ENTITIES_DIR / 'Annotations'
SENT_DIR     = ENTITIES_DIR / 'Sentences'
assert ANN_DIR.exists(), f'Annotations not found at {ANN_DIR}'
print(f'Annotations: {len(list(ANN_DIR.iterdir())):,} XML files')

/Users/siddharthraj/Documents/my-projects/region-grounded/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cpu
Annotations: 31,783 XML files


In [2]:
# ── 2. Load Flickr30k parquet + Entities annotations ─────────────────────────
import glob

# Use parquet shards from HuggingFace cache (already downloaded)
HF_CACHE = Path.home() / '.cache' / 'huggingface' / 'hub'
pq_files = sorted(glob.glob(
    str(HF_CACHE / 'datasets--nlphuji--flickr30k' / '**' / '*.parquet'),
    recursive=True
))
assert pq_files, 'No cached parquet shards found — run flickr-30-analysis.ipynb first'
print(f'Found {len(pq_files)} parquet shards')

hf_data = load_dataset('parquet', data_files={'data': pq_files})['data']
print(f'Total rows: {len(hf_data):,}')

# Filter to val split
val_rows = [r for r in hf_data if r['split'] == 'val']
print(f'Val rows  : {len(val_rows):,}')

# Import Entities utils
if str(ENTITIES_DIR) not in sys.path:
    sys.path.insert(0, str(ENTITIES_DIR))
from flickr30k_entities_utils import get_sentence_data, get_annotations


def load_phrase_boxes(row):
    """Return list of (phrase_str, [x1,y1,x2,y2]) for one dataset row."""
    stem = row['filename'].replace('.jpg', '')
    try:
        anns  = get_annotations(str(ANN_DIR / f'{stem}.xml'))
        sents = get_sentence_data(str(SENT_DIR / f'{stem}.txt'))
    except Exception:
        return []
    pairs, seen = [], set()
    for sent in sents:
        for phrase in sent['phrases']:
            pid = phrase['phrase_id']
            if pid in seen or pid not in anns['boxes']:
                continue
            seen.add(pid)
            for box in anns['boxes'][pid]:
                pairs.append((phrase['phrase'], box))
    return pairs


def get_pil(row):
    img = row['image']
    if isinstance(img, PILImage.Image):
        return img.convert('RGB')
    return PILImage.open(io.BytesIO(img['bytes'])).convert('RGB')


# Quick sanity check
sample = val_rows[0]
pb = load_phrase_boxes(sample)
print(f'\nSample: {sample["filename"]}  phrase-bbox pairs: {len(pb)}')
print(f'  e.g. {pb[0] if pb else "none"}')

Found 9 parquet shards
Total rows: 31,014
Val rows  : 1,014

Sample: 1018148011.jpg  phrase-bbox pairs: 6
  e.g. ('A group of people', [85, 35, 213, 106])


In [3]:
# ── 3. Load frozen SigLIP-B/16-384 ───────────────────────────────────────────
processor = AutoProcessor.from_pretrained(CFG['model_id'])
model     = SiglipModel.from_pretrained(CFG['model_id']).to(DEVICE).eval()

# Freeze everything
for p in model.parameters():
    p.requires_grad_(False)

total = sum(p.numel() for p in model.parameters())
print(f'Loaded {CFG["model_id"]}  ({total/1e6:.0f}M params, frozen)')

Loaded google/siglip-base-patch16-384  (203M params, frozen)


In [4]:
# ── 4. Pointing game evaluation ───────────────────────────────────────────────

def _maskclip_attn_fwd(self, hidden_states, attention_mask=None, **kwargs):
    """Replace full self-attention with value-only projection (MaskCLIP bypass)."""
    return self.out_proj(self.v_proj(hidden_states)), None


def pointing_game_eval(model, processor, val_rows, cfg,
                        device=DEVICE, n_images=200, seed=42):
    """
    Flickr30k Entities Pointing Game.

    For each (phrase, bbox) pair in the sampled val images:
      1. MaskCLIP-style patch features from last vision layer
      2. Phrase encoded via text model (EOS token)
      3. Argmax over patch-phrase cosine similarities
      4. Map argmax patch centre back to original image coords
      5. Hit if centre falls inside GT bbox

    Returns: accuracy (%), n_correct, n_total, per-image breakdown list
    """
    random.seed(seed)
    rows = random.sample(val_rows, min(n_images, len(val_rows)))

    # Apply MaskCLIP bypass on the last vision attention layer
    last_attn = model.vision_model.encoder.layers[-1].self_attn
    orig_fwd  = last_attn.forward
    last_attn.forward = types.MethodType(_maskclip_attn_fwd, last_attn)

    n_correct = 0
    n_total   = 0
    breakdown = []   # (filename, n_pairs, n_correct_for_image)

    n_side     = cfg['n_side']        # 24
    eval_size  = cfg['eval_size']     # 384
    patch_px   = eval_size / n_side   # 16.0

    try:
        for row in tqdm(rows, desc='Pointing game'):
            phrase_boxes = load_phrase_boxes(row)
            if not phrase_boxes:
                continue

            pil  = get_pil(row)
            W, H = pil.size

            # ── Vision: patch features (N, D) ────────────────────────────────
            pix = processor(images=pil, return_tensors='pt').pixel_values.to(device)
            with torch.no_grad():
                out = model.vision_model(pixel_values=pix)
            # last_hidden_state after post_layernorm; MaskCLIP bypass already applied
            patch_feats = F.normalize(out.last_hidden_state[0], dim=-1)  # (N, D)

            img_correct = 0

            for phrase, bbox in phrase_boxes:
                x1, y1, x2, y2 = bbox

                # Skip degenerate boxes
                if x2 <= x1 or y2 <= y1:
                    continue

                # ── Text: EOS token embedding (D,) ───────────────────────────
                txt_in = processor(
                    text=[phrase], return_tensors='pt',
                    padding=True, truncation=True
                ).to(device)
                with torch.no_grad():
                    txt_hs = model.text_model(**txt_in).last_hidden_state  # (1, T, D)
                txt_feat = F.normalize(txt_hs[0, -1, :], dim=-1)           # (D,)

                # ── Similarity + argmax ───────────────────────────────────────
                sim      = patch_feats @ txt_feat          # (N,)
                best_idx = sim.argmax().item()

                # Argmax patch centre in original pixel coords
                p_row = best_idx // n_side
                p_col = best_idx % n_side
                cx = (p_col + 0.5) * patch_px * W / eval_size
                cy = (p_row + 0.5) * patch_px * H / eval_size

                hit = (x1 <= cx <= x2) and (y1 <= cy <= y2)
                n_correct   += int(hit)
                img_correct += int(hit)
                n_total     += 1

            breakdown.append((row['filename'], len(phrase_boxes), img_correct))

    finally:
        last_attn.forward = orig_fwd   # always restore

    accuracy = 100.0 * n_correct / max(n_total, 1)
    return accuracy, n_correct, n_total, breakdown


print('pointing_game_eval defined.')

pointing_game_eval defined.


In [5]:
# ── 5. Run on 200 val images ──────────────────────────────────────────────────
acc, n_correct, n_total, breakdown = pointing_game_eval(
    model, processor, val_rows,
    cfg      = CFG,
    device   = DEVICE,
    n_images = CFG['n_images'],
    seed     = CFG['seed'],
)

print('=' * 50)
print(f'  Flickr30k Pointing Game — frozen SigLIP-B/16-384')
print(f'  Val images evaluated : {len(breakdown)}')
print(f'  Phrase-bbox pairs    : {n_total}')
print(f'  Correct hits         : {n_correct}')
print(f'  Pointing Game Acc.   : {acc:.2f}%')
print('=' * 50)

# Per-image breakdown (worst 5)
breakdown_sorted = sorted(breakdown, key=lambda x: x[2] / max(x[1], 1))
print('\nWorst 5 images (by per-image accuracy):')
for fname, n_pairs, n_hit in breakdown_sorted[:5]:
    print(f'  {fname}  {n_hit}/{n_pairs} ({100*n_hit/max(n_pairs,1):.0f}%)')

print('\nBest 5 images:')
for fname, n_pairs, n_hit in breakdown_sorted[-5:][::-1]:
    print(f'  {fname}  {n_hit}/{n_pairs} ({100*n_hit/max(n_pairs,1):.0f}%)')

Pointing game: 100%|██████████| 200/200 [00:50<00:00,  3.96it/s]

  Flickr30k Pointing Game — frozen SigLIP-B/16-384
  Val images evaluated : 200
  Phrase-bbox pairs    : 2124
  Correct hits         : 313
  Pointing Game Acc.   : 14.74%

Worst 5 images (by per-image accuracy):
  289625522.jpg  0/5 (0%)
  481054596.jpg  0/13 (0%)
  6371136393.jpg  0/19 (0%)
  2579268572.jpg  0/7 (0%)
  4407490214.jpg  0/12 (0%)

Best 5 images:
  1752454466.jpg  4/4 (100%)
  3578841731.jpg  3/4 (75%)
  1206506157.jpg  3/4 (75%)
  210625425.jpg  3/4 (75%)
  4678723492.jpg  4/6 (67%)


In [6]:
# ── 6. Recall@1 at IoU ≥ 0.5 ─────────────────────────────────────────────────
# For each (phrase, bbox) pair:
#   1. Same patch features + phrase embedding as pointing game
#   2. Take the top-k patches by similarity score
#   3. Form the tight bounding box around those patches in original image coords
#   4. Compute IoU with GT bbox — hit if IoU >= 0.5
#
# We try three selection strategies and report all three:
#   - top10   : fixed 10 patches (out of 576, ~1.7% of image)
#   - top25   : fixed 25 patches (~4.3%)
#   - halfmax : all patches with sim >= 0.5 * max_sim (adaptive)

def patches_to_box(patch_indices, n_side, patch_px, W, H, eval_size):
    """Convert a list of flat patch indices to a tight bbox in original image coords."""
    rows = np.array([i // n_side for i in patch_indices])
    cols = np.array([i %  n_side for i in patch_indices])
    x1 = float(cols.min()       * patch_px * W / eval_size)
    y1 = float(rows.min()       * patch_px * H / eval_size)
    x2 = float((cols.max() + 1) * patch_px * W / eval_size)
    y2 = float((rows.max() + 1) * patch_px * H / eval_size)
    return x1, y1, x2, y2


def iou(box_a, box_b):
    """IoU of two (x1,y1,x2,y2) boxes."""
    ax1, ay1, ax2, ay2 = box_a
    bx1, by1, bx2, by2 = box_b
    ix1, iy1 = max(ax1, bx1), max(ay1, by1)
    ix2, iy2 = min(ax2, bx2), min(ay2, by2)
    inter = max(0, ix2 - ix1) * max(0, iy2 - iy1)
    if inter == 0:
        return 0.0
    union = (ax2-ax1)*(ay2-ay1) + (bx2-bx1)*(by2-by1) - inter
    return inter / union if union > 0 else 0.0


def recall_at1_eval(model, processor, val_rows, cfg,
                    device=DEVICE, n_images=200, seed=42,
                    iou_thresh=0.5):
    """
    Flickr30k Entities Recall@1 at IoU >= iou_thresh.
    Returns dict with results for three patch-selection strategies.
    """
    random.seed(seed)
    rows = random.sample(val_rows, min(n_images, len(val_rows)))

    last_attn = model.vision_model.encoder.layers[-1].self_attn
    orig_fwd  = last_attn.forward
    last_attn.forward = types.MethodType(_maskclip_attn_fwd, last_attn)

    n_side    = cfg['n_side']
    eval_size = cfg['eval_size']
    patch_px  = eval_size / n_side

    hits   = {'top10': 0, 'top25': 0, 'halfmax': 0}
    totals = {'top10': 0, 'top25': 0, 'halfmax': 0}

    try:
        for row in tqdm(rows, desc='Recall@1'):
            phrase_boxes = load_phrase_boxes(row)
            if not phrase_boxes:
                continue

            pil  = get_pil(row)
            W, H = pil.size

            pix = processor(images=pil, return_tensors='pt').pixel_values.to(device)
            with torch.no_grad():
                out = model.vision_model(pixel_values=pix)
            patch_feats = F.normalize(out.last_hidden_state[0], dim=-1)  # (N, D)

            for phrase, gt_bbox in phrase_boxes:
                x1, y1, x2, y2 = gt_bbox
                if x2 <= x1 or y2 <= y1:
                    continue

                txt_in = processor(
                    text=[phrase], return_tensors='pt',
                    padding=True, truncation=True
                ).to(device)
                with torch.no_grad():
                    txt_hs = model.text_model(**txt_in).last_hidden_state
                txt_feat = F.normalize(txt_hs[0, -1, :], dim=-1)

                sim = (patch_feats @ txt_feat).cpu().numpy()  # (N,)

                strategies = {
                    'top10':   np.argsort(sim)[-10:][::-1],
                    'top25':   np.argsort(sim)[-25:][::-1],
                    'halfmax': np.where(sim >= 0.5 * sim.max())[0],
                }

                for name, sel_idx in strategies.items():
                    if len(sel_idx) == 0:
                        sel_idx = [int(sim.argmax())]
                    pred_box = patches_to_box(
                        sel_idx, n_side, patch_px, W, H, eval_size
                    )
                    score = iou(pred_box, (x1, y1, x2, y2))
                    hits[name]   += int(score >= iou_thresh)
                    totals[name] += 1

    finally:
        last_attn.forward = orig_fwd

    results = {k: 100.0 * hits[k] / max(totals[k], 1) for k in hits}
    return results, hits, totals


recall_results, recall_hits, recall_totals = recall_at1_eval(
    model, processor, val_rows,
    cfg      = CFG,
    device   = DEVICE,
    n_images = CFG['n_images'],
    seed     = CFG['seed'],
    iou_thresh = 0.5,
)

print('=' * 55)
print(f'  Flickr30k Recall@1 (IoU ≥ 0.5) — frozen SigLIP-B/16-384')
print(f'  Val images: {CFG["n_images"]}   Pairs: {recall_totals["top10"]}')
print('=' * 55)
print(f'  Patch selection    Hits   Total   Recall@1')
print(f'  ─────────────────────────────────────────')
for name in ['top10', 'top25', 'halfmax']:
    print(f'  {name:<18} {recall_hits[name]:<6} {recall_totals[name]:<7} {recall_results[name]:.2f}%')
print('=' * 55)
print()
print(f'  Pointing Game Acc. (from cell 5): 14.74%')
print(f'  (shown for comparison — same 200 images, seed=42)')

Recall@1: 100%|██████████| 200/200 [01:04<00:00,  3.10it/s]

  Flickr30k Recall@1 (IoU ≥ 0.5) — frozen SigLIP-B/16-384
  Val images: 200   Pairs: 2124
  Patch selection    Hits   Total   Recall@1
  ─────────────────────────────────────────
  top10              145    2124    6.83%
  top25              168    2124    7.91%
  halfmax            161    2124    7.58%

  Pointing Game Acc. (from cell 5): 14.74%
  (shown for comparison — same 200 images, seed=42)
